
# Phase 3 v4 — Hybrid Forecasting with Phase 2 Backbone, Calibrated Hazard Overlays, Maria Backtest, and Regional Analysis

This notebook uses the **best Phase 2 model logic** as the structural forecasting backbone and adds **explicit disaster-shock overlays** for scenario simulation.

## Main design
- **Baseline structural forecast**: uses the validated Phase 2 feature set and a geography-aware Extra Trees model.
- **Scenario layer**: applies calibrated hurricane / earthquake shock overlays on top of the structural baseline.
- **Calibration**: includes a Maria-like backtest path for island and regions.
- **Scales**: municipal, regional, and island outputs.
- **Charts**: expanded set of comparison, impact, ladder, and backtest charts.


In [1]:

import os, json, warnings, math
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

warnings.filterwarnings("ignore")

np.random.seed(42)


In [2]:

# ----------------------------
# Output folder
# ----------------------------
OUTPUT_DIR = Path("phase3_outputs_v4")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def save_csv(df, name):
    path = OUTPUT_DIR / name
    df.to_csv(path, index=False)
    print(f"Saved CSV: {path}")
    return str(path)

def save_json(obj, name):
    path = OUTPUT_DIR / name
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False)
    print(f"Saved JSON: {path}")
    return str(path)

def save_fig(fig, name, dpi=150):
    path = OUTPUT_DIR / name
    fig.savefig(path, bbox_inches="tight", dpi=dpi)
    print(f"Saved FIG: {path}")
    plt.close(fig)
    return str(path)

saved_files = []
chart_index_records = []


In [3]:

# ----------------------------
# File paths
# ----------------------------
preferred_files = [
    "processed_puerto_rico_data_rebuild.csv",
    "processed_puerto_rico_data_enriched.csv",
]

data_path = None
for p in preferred_files:
    if Path(p).exists():
        data_path = Path(p)
        break

if data_path is None:
    raise FileNotFoundError(
        "Could not find processed_puerto_rico_data_rebuild.csv or processed_puerto_rico_data_enriched.csv in the current directory."
    )

print("Using data file:", data_path)
df = pd.read_csv(data_path)
print(df.shape)
df.head()


Using data file: processed_puerto_rico_data_rebuild.csv
(1170, 160)


,year,municipio,total_population,total_population_moe,total_population_16plus,total_population_16plus_moe,total_housing_units,total_housing_units_moe,median_income_nominal,median_income_real,...,wind_3yr_x_svi_current,seismic_3yr_x_svi_current,years_since_hurricane,years_since_earthquake,years_since_hurricane_capped_5,years_since_earthquake_capped_5,target_pop_change_1y_winsor_1_99,target_pop_change_2y_avg_winsor_1_99,target_pop_change_3y_avg_winsor_1_99,municipality_id
0,2010,Adjuntas,19541.0,0,14834,99,6548,133,11983.0,16743.095245,...,0.063443,3055.178423,0,0,0,0,-0.138171,-0.212600,-0.354358,0
1,2011,Adjuntas,19514.0,0,15000,107,6834,161,12975.0,17577.288950,...,0.550670,4292.995237,0,0,0,0,-0.286973,-0.462276,-0.559995,0
2,2012,Adjuntas,19458.0,0,15081,80,7037,114,13095.0,17379.542368,...,0.550670,4862.171141,1,0,1,0,-0.637270,-0.696226,-0.857017,0
3,2013,Adjuntas,19334.0,0,15087,79,7298,140,11528.0,15078.786201,...,0.645874,2495.720963,0,0,0,0,-0.755146,-0.966708,-0.999579,0
4,2014,Adjuntas,19188.0,0,15080,88,7514,136,10550.0,13580.165846,...,0.265284,6766.905850,0,0,0,0,-1.177819,-1.121570,-1.165288,0


In [4]:

# ----------------------------
# Region map
# ----------------------------
region_map = {
    # Metro
    "San Juan": "Metro", "Bayamón": "Metro", "Carolina": "Metro", "Cataño": "Metro",
    "Guaynabo": "Metro", "Toa Alta": "Metro", "Toa Baja": "Metro", "Trujillo Alto": "Metro",
    # North
    "Arecibo": "North", "Barceloneta": "North", "Camuy": "North", "Dorado": "North",
    "Florida": "North", "Hatillo": "North", "Manatí": "North", "Quebradillas": "North",
    "Vega Alta": "North", "Vega Baja": "North",
    # South
    "Arroyo": "South", "Coamo": "South", "Guayama": "South", "Guayanilla": "South",
    "Juana Díaz": "South", "Patillas": "South", "Peñuelas": "South", "Ponce": "South",
    "Salinas": "South", "Santa Isabel": "South", "Villalba": "South", "Yauco": "South",
    # West
    "Aguada": "West", "Aguadilla": "West", "Añasco": "West", "Cabo Rojo": "West",
    "Guánica": "West", "Hormigueros": "West", "Isabela": "West", "Lajas": "West",
    "Las Marías": "West", "Maricao": "West", "Mayagüez": "West", "Moca": "West",
    "Rincón": "West", "Sabana Grande": "West", "San Germán": "West", "San Sebastián": "West",
    # East
    "Canóvanas": "East", "Ceiba": "East", "Fajardo": "East", "Humacao": "East",
    "Juncos": "East", "Las Piedras": "East", "Loíza": "East", "Luquillo": "East",
    "Maunabo": "East", "Naguabo": "East", "Río Grande": "East", "San Lorenzo": "East",
    "Yabucoa": "East", "Caguas": "East", "Gurabo": "East", "Culebra": "East", "Vieques": "East",
    # Central Mountains
    "Adjuntas": "Central Mountains", "Aguas Buenas": "Central Mountains", "Aibonito": "Central Mountains",
    "Barranquitas": "Central Mountains", "Cayey": "Central Mountains", "Ciales": "Central Mountains",
    "Cidra": "Central Mountains", "Comerío": "Central Mountains", "Corozal": "Central Mountains",
    "Jayuya": "Central Mountains", "Lares": "Central Mountains", "Morovis": "Central Mountains",
    "Naranjito": "Central Mountains", "Orocovis": "Central Mountains", "Utuado": "Central Mountains",
}


In [5]:

# ----------------------------
# Basic cleanup / canonical columns
# ----------------------------
rename_candidates = {
    "municipality": "municipio",
    "Municipio": "municipio",
    "Municipality": "municipio",
    "Region": "region",
    "Year": "year",
    "total_pop": "total_population",
    "population": "total_population",
}

for old, new in rename_candidates.items():
    if old in df.columns and new not in df.columns:
        df = df.rename(columns={old: new})

if "municipio" not in df.columns:
    raise KeyError("Expected a 'municipio' column.")
if "year" not in df.columns:
    raise KeyError("Expected a 'year' column.")
if "total_population" not in df.columns:
    # fallback: choose a likely population column
    pop_candidates = [c for c in df.columns if "population" in c.lower()]
    if not pop_candidates:
        raise KeyError("Could not identify total population column.")
    df["total_population"] = pd.to_numeric(df[pop_candidates[0]], errors="coerce")

df["municipio"] = df["municipio"].astype(str).str.strip()
if "region" not in df.columns:
    df["region"] = df["municipio"].map(region_map)
else:
    df["region"] = df["region"].fillna(df["municipio"].map(region_map))

df["year"] = pd.to_numeric(df["year"], errors="coerce").astype(int)
df["total_population"] = pd.to_numeric(df["total_population"], errors="coerce")
df = df.sort_values(["municipio", "year"]).reset_index(drop=True)

print(df[["municipio","region","year","total_population"]].head())
print(df.shape)


  municipio             region  year  total_population
0  Adjuntas  Central Mountains  2010           19541.0
1  Adjuntas  Central Mountains  2011           19514.0
2  Adjuntas  Central Mountains  2012           19458.0
3  Adjuntas  Central Mountains  2013           19334.0
4  Adjuntas  Central Mountains  2014           19188.0
(1170, 161)


In [6]:

# ----------------------------
# Ensure required target / feature columns exist
# ----------------------------
g = df.groupby("municipio")["total_population"]

if "target_pop_change_1y" not in df.columns:
    df["target_pop_change_1y"] = ((g.shift(-1) / df["total_population"]) - 1) * 100

if "target_pop_change_3y_avg" not in df.columns:
    df["target_pop_change_3y_avg"] = (((g.shift(-3) / df["total_population"]) ** (1/3)) - 1) * 100

if "lag_pop_change_1" not in df.columns:
    df["lag_pop_change_1"] = g.pct_change() * 100
if "lag_pop_change_2" not in df.columns:
    df["lag_pop_change_2"] = df.groupby("municipio")["lag_pop_change_1"].shift(1)

# time since event defaults if absent
for col in ["years_since_hurricane","years_since_earthquake","post_maria","post_earthquake_2020",
            "wind_3yr_sum","seismic_3yr_sum","wind_3yr_x_svi","seismic_3yr_x_svi",
            "svi_pca_1","income_growth","establishment_growth","lag_total_crime_rate"]:
    if col not in df.columns:
        df[col] = 0.0

# region fallback
df["region"] = df["region"].fillna("Unknown")


In [7]:

# ----------------------------
# Phase 2 best-model feature set
# ----------------------------
PHASE2_FEATURES = [
    "lag_pop_change_1",
    "lag_pop_change_2",
    "svi_pca_1",
    "wind_3yr_sum",
    "seismic_3yr_sum",
    "wind_3yr_x_svi",
    "seismic_3yr_x_svi",
    "income_growth",
    "establishment_growth",
    "lag_total_crime_rate",
    "year",
    "municipio",
    "region",
]

missing = [c for c in PHASE2_FEATURES if c not in df.columns]
if missing:
    raise KeyError(f"Missing Phase 2 features: {missing}")

print("Phase 2 backbone features available.")


Phase 2 backbone features available.


In [8]:
# ----------------------------
# Load the exact fitted Phase 2 backbone if available.
# Fallback: reconstruct the Phase 2 backbone inside this notebook.
# Target: 3-year average forward population change
# ----------------------------
model_df = df.dropna(subset=["target_pop_change_3y_avg"]).copy()

available_years = sorted(model_df["year"].unique())
print("Available target years:", available_years)

# Use the exact Phase 2 split logic for strict reproducibility:
# train <= 2018, val == 2019, test >= 2020
train_mask = model_df["year"] <= 2018
val_mask = model_df["year"] == 2019
test_mask = model_df["year"] >= 2020

train_df = model_df.loc[train_mask].copy()
val_df = model_df.loc[val_mask].copy()
test_df = model_df.loc[test_mask].copy()

train_years = sorted(train_df["year"].unique())
val_years = sorted(val_df["year"].unique())
test_years = sorted(test_df["year"].unique())

print("Train years:", train_years)
print("Val years:", val_years)
print("Test years:", test_years)
print("Split shapes:", train_df.shape, val_df.shape, test_df.shape)

# Build the strict Phase 2-style feature matrices
X_train = train_df[PHASE2_FEATURES].copy()
y_train = train_df["target_pop_change_3y_avg"].copy()
X_val = val_df[PHASE2_FEATURES].copy()
y_val = val_df["target_pop_change_3y_avg"].copy()
X_test = test_df[PHASE2_FEATURES].copy()
y_test = test_df["target_pop_change_3y_avg"].copy()

def metric_block(y_true, y_pred):
    return {
        "r2": float(r2_score(y_true, y_pred)),
        "rmse": float(np.sqrt(mean_squared_error(y_true, y_pred))),
        "mae": float(mean_absolute_error(y_true, y_pred)),
    }

PHASE2_DIR = Path("phase2_outputs_v4")
phase2_pipeline_path = PHASE2_DIR / "phase2_v4_best_pipeline.joblib"
reference_phase2_path = PHASE2_DIR / "phase2_v4_best_model_summary.json"
phase2_pipeline_loaded = False

if phase2_pipeline_path.exists():
    phase2_backbone_model = joblib.load(phase2_pipeline_path)
    phase2_pipeline_loaded = True
    print(f"Loaded exact fitted Phase 2 pipeline: {phase2_pipeline_path}")
else:
    print(f"No fitted Phase 2 pipeline found at {phase2_pipeline_path}; reconstructing backbone inside Phase 3.")
    num_cols = [c for c in PHASE2_FEATURES if c not in ["municipio", "region"]]
    cat_cols = ["municipio", "region"]

    try:
        ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        ohe = OneHotEncoder(handle_unknown="ignore", sparse=False)

    preprocessor = ColumnTransformer(
        transformers=[
            ("num", Pipeline([
                ("imputer", SimpleImputer(strategy="median"))
            ]), num_cols),
            ("cat", Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("ohe", ohe)
            ]), cat_cols),
        ],
        remainder="drop"
    )

    phase2_backbone_model = Pipeline([
        ("prep", preprocessor),
        ("model", ExtraTreesRegressor(
            n_estimators=500,
            random_state=42,
            min_samples_leaf=2,
            n_jobs=-1
        ))
    ])

    phase2_backbone_model.fit(X_train, y_train)

val_pred = phase2_backbone_model.predict(X_val)
test_pred = phase2_backbone_model.predict(X_test)

phase2_reference = None
if reference_phase2_path.exists():
    with open(reference_phase2_path, "r", encoding="utf-8") as f:
        phase2_reference = json.load(f)
    print(f"Loaded Phase 2 reference summary: {reference_phase2_path}")
else:
    print(f"No saved Phase 2 summary found at {reference_phase2_path}")

backbone_summary = {
    "model": "extra_trees",
    "target": "target_pop_change_3y_avg",
    "features": PHASE2_FEATURES,
    "n_train": int(len(train_df)),
    "n_val": int(len(val_df)),
    "n_test": int(len(test_df)),
    "metrics": {
        "validation": metric_block(y_val, val_pred),
        "test": metric_block(y_test, test_pred),
    },
    "phase2_pipeline_loaded": phase2_pipeline_loaded,
    "reference_phase2_summary_loaded": bool(phase2_reference is not None),
}

if phase2_reference is not None:
    ref_val_r2 = phase2_reference.get("val_r2")
    ref_test_r2 = phase2_reference.get("test_r2")
    backbone_summary["phase2_reference_metrics"] = {
        "validation_r2": ref_val_r2,
        "test_r2": ref_test_r2
    }
    backbone_summary["reproduction_gap"] = {
        "validation_r2_gap": None if ref_val_r2 is None else float(backbone_summary["metrics"]["validation"]["r2"] - ref_val_r2),
        "test_r2_gap": None if ref_test_r2 is None else float(backbone_summary["metrics"]["test"]["r2"] - ref_test_r2),
    }

backbone_summary


Available target years: [np.int64(2010), np.int64(2011), np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021)]
Train years: [np.int64(2010), np.int64(2011), np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018)]
Val years: [np.int64(2019)]
Test years: [np.int64(2020), np.int64(2021)]
Split shapes: (702, 161) (78, 161) (156, 161)
Loaded exact fitted Phase 2 pipeline: phase2_outputs_v4/phase2_v4_best_pipeline.joblib
Loaded Phase 2 reference summary: phase2_outputs_v4/phase2_v4_best_model_summary.json


{'model': 'extra_trees',
 'target': 'target_pop_change_3y_avg',
 'features': ['lag_pop_change_1',
  'lag_pop_change_2',
  'svi_pca_1',
  'wind_3yr_sum',
  'seismic_3yr_sum',
  'wind_3yr_x_svi',
  'seismic_3yr_x_svi',
  'income_growth',
  'establishment_growth',
  'lag_total_crime_rate',
  'year',
  'municipio',
  'region'],
 'n_train': 702,
 'n_val': 78,
 'n_test': 156,
 'metrics': {'validation': {'r2': 0.7469719857441863,
   'rmse': 0.8924329399791264,
   'mae': 0.6942979737824501},
  'test': {'r2': 0.23405939583721558,
   'rmse': 1.3162361588476064,
   'mae': 1.0161787502970374}},
 'phase2_pipeline_loaded': True,
 'reference_phase2_summary_loaded': True,
 'phase2_reference_metrics': {'validation_r2': 0.4079199696004543,
  'test_r2': 0.17858057969069385},
 'reproduction_gap': {'validation_r2_gap': 0.33905201614373204,
  'test_r2_gap': 0.055478816146521726}}

In [9]:

saved_files.append(save_json(backbone_summary, "phase3_v4_backbone_model_summary.json"))


Saved JSON: phase3_outputs_v4/phase3_v4_backbone_model_summary.json


In [10]:
# Optional: compare reconstructed Phase 3 backbone with saved Phase 2 reference metrics
if backbone_summary.get("reference_phase2_summary_loaded", False):
    comparison_rows = [
        {
            "split": "validation",
            "phase2_reference_r2": backbone_summary["phase2_reference_metrics"]["validation_r2"],
            "phase3_reconstructed_r2": backbone_summary["metrics"]["validation"]["r2"],
            "gap": backbone_summary["reproduction_gap"]["validation_r2_gap"],
        },
        {
            "split": "test",
            "phase2_reference_r2": backbone_summary["phase2_reference_metrics"]["test_r2"],
            "phase3_reconstructed_r2": backbone_summary["metrics"]["test"]["r2"],
            "gap": backbone_summary["reproduction_gap"]["test_r2_gap"],
        },
    ]
    backbone_comparison_df = pd.DataFrame(comparison_rows)
    saved_files.append(save_csv(backbone_comparison_df, "phase3_v4_backbone_vs_phase2_reference.csv"))
    display(backbone_comparison_df)
else:
    print("No saved Phase 2 summary found in phase2_outputs_v4; skipping backbone comparison.")

Saved CSV: phase3_outputs_v4/phase3_v4_backbone_vs_phase2_reference.csv


,split,phase2_reference_r2,phase3_reconstructed_r2,gap
0,validation,0.407920,0.746972,0.339052
1,test,0.178581,0.234059,0.055479


## Backbone-model reproducibility note

This notebook now **loads the exact fitted Phase 2 pipeline artifact when available**.  
If `phase2_outputs_v4/phase2_v4_best_pipeline.joblib` is present, Phase 3 uses that artifact as the structural forecasting backbone. If it is not present, the notebook falls back to reconstructing the Extra Trees backbone inside Phase 3.

If `phase2_outputs_v4/phase2_v4_best_model_summary.json` is also present in the same folder, the notebook compares the Phase 3 validation/test performance with the saved Phase 2 reference so you can see whether the forecasting engine is closely reproducing the original Phase 2 result.

If the loaded artifact is unavailable and the reconstructed Phase 3 backbone differs materially from the saved Phase 2 summary, treat the scenario forecasts as **conditional on this reconstructed backbone**, not as exact continuations of the Phase 2 model.


In [11]:

# ----------------------------
# Baseline annualized forecast builder
# We convert model's 3-year average pct change prediction into an annualized path.
# ----------------------------
history_df = df.copy()
latest_year = int(history_df["year"].max())
latest_rows = history_df.sort_values("year").groupby("municipio").tail(1).copy()
latest_rows = latest_rows.sort_values("municipio").reset_index(drop=True)

print("Latest year:", latest_year)
print("Latest rows:", latest_rows.shape)

# Municipio historical volatility used for uncertainty bands and shock heterogeneity
muni_vol = (
    model_df.groupby("municipio")["target_pop_change_1y"]
    .std()
    .fillna(model_df["target_pop_change_1y"].std())
    .to_dict()
)

global_vol = float(model_df["target_pop_change_1y"].std())
global_vol


Latest year: 2024
Latest rows: (78, 161)


1.9308983000926878

In [12]:

# ----------------------------
# Scenario definitions
# Hybrid design:
# 1) structural baseline path from Phase 2 backbone
# 2) explicit event-year / decay overlay calibrated for visible shock dynamics
# ----------------------------
HURRICANE_LADDER = {
    "baseline": {"kind": "none"},
    "hurricane_cat1": {"kind": "hurricane", "category": 1, "event_year": 2026},
    "hurricane_cat2": {"kind": "hurricane", "category": 2, "event_year": 2026},
    "hurricane_cat3": {"kind": "hurricane", "category": 3, "event_year": 2026},
    "hurricane_cat4": {"kind": "hurricane", "category": 4, "event_year": 2026},
    "hurricane_cat5_maria_like": {"kind": "hurricane", "category": 5, "event_year": 2026},
}

EARTHQUAKE_LADDER = {
    "earthquake_m6_0": {"kind": "earthquake", "magnitude": 6.0, "event_year": 2026},
    "earthquake_m6_5": {"kind": "earthquake", "magnitude": 6.5, "event_year": 2026},
    "earthquake_m6_8": {"kind": "earthquake", "magnitude": 6.8, "event_year": 2026},
    "earthquake_m7_2": {"kind": "earthquake", "magnitude": 7.2, "event_year": 2026},
    "earthquake_m7_5": {"kind": "earthquake", "magnitude": 7.5, "event_year": 2026},
}

scenario_definitions = {}
scenario_definitions.update(HURRICANE_LADDER)
scenario_definitions.update(EARTHQUAKE_LADDER)

saved_files.append(save_json(scenario_definitions, "phase3_v4_scenario_definitions.json"))
scenario_definitions


Saved JSON: phase3_outputs_v4/phase3_v4_scenario_definitions.json


{'baseline': {'kind': 'none'},
 'hurricane_cat1': {'kind': 'hurricane', 'category': 1, 'event_year': 2026},
 'hurricane_cat2': {'kind': 'hurricane', 'category': 2, 'event_year': 2026},
 'hurricane_cat3': {'kind': 'hurricane', 'category': 3, 'event_year': 2026},
 'hurricane_cat4': {'kind': 'hurricane', 'category': 4, 'event_year': 2026},
 'hurricane_cat5_maria_like': {'kind': 'hurricane',
  'category': 5,
  'event_year': 2026},
 'earthquake_m6_0': {'kind': 'earthquake',
  'magnitude': 6.0,
  'event_year': 2026},
 'earthquake_m6_5': {'kind': 'earthquake',
  'magnitude': 6.5,
  'event_year': 2026},
 'earthquake_m6_8': {'kind': 'earthquake',
  'magnitude': 6.8,
  'event_year': 2026},
 'earthquake_m7_2': {'kind': 'earthquake',
  'magnitude': 7.2,
  'event_year': 2026},
 'earthquake_m7_5': {'kind': 'earthquake',
  'magnitude': 7.5,
  'event_year': 2026}}

### Scenario-design note

The hurricane and earthquake scenarios in this notebook are **stylized scenario overlays**.  
They are designed to create transparent and interpretable stress tests on top of the structural baseline forecast. They should **not** be interpreted as estimated causal effects of specific future disasters.

In [13]:

# ----------------------------
# Shock calibration helpers
# These are explicit overlays on top of the Phase 2 baseline engine.
# The scale is in ANNUAL percent-population-change adjustments.
# ----------------------------
REGION_HURRICANE_MULT = {
    "East": 1.20,
    "South": 1.10,
    "Metro": 1.00,
    "Central Mountains": 1.05,
    "North": 0.95,
    "West": 1.00,
    "Unknown": 1.00,
}
REGION_EARTHQUAKE_MULT = {
    "South": 1.25,
    "West": 1.15,
    "Central Mountains": 1.05,
    "Metro": 0.95,
    "East": 0.95,
    "North": 0.90,
    "Unknown": 1.00,
}

def hurricane_base_annual_adjustment(category):
    # Explicit one-year shock followed by decay; tuned to produce visible response.
    return {1: -0.35, 2: -0.70, 3: -1.20, 4: -1.90, 5: -2.80}.get(category, -1.0)

def earthquake_base_annual_adjustment(magnitude):
    if magnitude < 6.0:
        return -0.20
    if magnitude < 6.5:
        return -0.45
    if magnitude < 6.8:
        return -0.75
    if magnitude < 7.2:
        return -1.05
    return -1.35

def shock_decay_profile(kind):
    # Event year, +1, +2 annual adjustment multipliers
    if kind == "hurricane":
        return [1.00, 0.60, 0.30]
    if kind == "earthquake":
        return [1.00, 0.45, 0.20]
    return [0.0, 0.0, 0.0]

def compute_maria_backtest_shock(event_year):
    # Maria-like backtest starts at 2017, stronger than Cat 4, with explicit recovery decay
    return {"kind": "hurricane", "category": 5, "event_year": event_year, "label": "maria_like_backtest"}

def apply_scenario_feature_overlay(base_row, year, scenario):
    row = base_row.copy()
    kind = scenario.get("kind", "none")
    event_year = scenario.get("event_year", 9999)

    # zero temporary shock inputs each year, then apply if in pulse window
    row["scenario_shock_event"] = 0.0
    row["scenario_shock_intensity"] = 0.0

    if kind == "none":
        return row, 0.0

    year_offset = year - event_year
    if year_offset < 0 or year_offset > 2:
        return row, 0.0

    decay = shock_decay_profile(kind)[year_offset]
    if kind == "hurricane":
        cat = scenario["category"]
        base_adj = hurricane_base_annual_adjustment(cat)
        region_mult = REGION_HURRICANE_MULT.get(row["region"], 1.0)
        svi_mult = 1.0 + max(0.0, float(row.get("svi_pca_1", 0.0))) * 0.08
        adj = base_adj * decay * region_mult * svi_mult

        row["wind_3yr_sum"] = float(row.get("wind_3yr_sum", 0.0)) + abs(base_adj) * 2.5 * decay
        row["wind_3yr_x_svi"] = float(row.get("wind_3yr_x_svi", 0.0)) + abs(base_adj) * max(0.0, float(row.get("svi_pca_1", 0.0))) * 2.0 * decay
        row["post_maria"] = 1.0
        row["years_since_hurricane"] = max(0, year - event_year)
        row["scenario_shock_event"] = 1.0 if year == event_year else 0.0
        row["scenario_shock_intensity"] = abs(base_adj) * decay
        return row, adj

    if kind == "earthquake":
        mag = scenario["magnitude"]
        base_adj = earthquake_base_annual_adjustment(mag)
        region_mult = REGION_EARTHQUAKE_MULT.get(row["region"], 1.0)
        svi_mult = 1.0 + max(0.0, float(row.get("svi_pca_1", 0.0))) * 0.06
        adj = base_adj * decay * region_mult * svi_mult

        row["seismic_3yr_sum"] = float(row.get("seismic_3yr_sum", 0.0)) + abs(base_adj) * 2.2 * decay
        row["seismic_3yr_x_svi"] = float(row.get("seismic_3yr_x_svi", 0.0)) + abs(base_adj) * max(0.0, float(row.get("svi_pca_1", 0.0))) * 1.7 * decay
        row["post_earthquake_2020"] = 1.0
        row["years_since_earthquake"] = max(0, year - event_year)
        row["scenario_shock_event"] = 1.0 if year == event_year else 0.0
        row["scenario_shock_intensity"] = abs(base_adj) * decay
        return row, adj

    return row, 0.0


In [14]:
# ----------------------------
# Forecast engine
# Baseline uses the reconstructed Phase 2 model.
# Scenario effects are stylized overlays on the annual structural annual growth rate.
# ----------------------------
FORECAST_YEARS = list(range(latest_year + 1, 2031))
N_SIMS = 200  # can reduce for testing
SAVE_DETAIL = False

# Use holdout residual scale for uncertainty (validation + test),
# which is more conservative than training residuals.
holdout_actual = np.concatenate([y_val.values, y_test.values])
holdout_pred = np.concatenate([val_pred, test_pred])
phase2_holdout_resid = holdout_actual - holdout_pred
residual_std = float(np.std(phase2_holdout_resid))
print("Residual std from holdout residuals:", residual_std)

def predicted_annual_pct_to_decimal(pct_3y_avg_annualized):
    """
    The Phase 2 target is already an annualized 3-year-average percent change.
    This helper only converts percent units to decimal units for simulation.
    Example: -1.2 (%) -> -0.012
    """
    return pct_3y_avg_annualized / 100.0

STATIC_DRIVER_COLS = [
    "svi_pca_1",
    "baseline_vulnerability_index",
    "municipio",
    "region"
]

def evolve_nonlag_covariates(sim_df, realized_pct, year, scenario):
    """
    Simple, transparent evolution rules for non-lag covariates.
    These are intentionally modest so baseline forecasts are not driven by
    fully static covariates. They introduce persistence + mean reversion
    rather than pretending to know future values exactly.
    """
    out = sim_df.copy()

    # smooth socioeconomic growth toward recent realized population change
    if "income_growth" in out.columns:
        out["income_growth"] = 0.65 * out["income_growth"].astype(float) + 0.15 * realized_pct
        out["income_growth"] = out["income_growth"].clip(-8, 8)

    if "establishment_growth" in out.columns:
        out["establishment_growth"] = 0.60 * out["establishment_growth"].astype(float) + 0.20 * realized_pct
        out["establishment_growth"] = out["establishment_growth"].clip(-10, 10)

    if "lag_total_crime_rate" in out.columns:
        out["lag_total_crime_rate"] = 0.80 * out["lag_total_crime_rate"].astype(float) + 0.20 * out["lag_total_crime_rate"].median()

    # rolling decay of hazard accumulators after the event pulse window
    for col, decay in [("wind_3yr_sum", 0.82), ("seismic_3yr_sum", 0.82), ("wind_3yr_x_svi", 0.82), ("seismic_3yr_x_svi", 0.82)]:
        if col in out.columns:
            out[col] = out[col].astype(float) * decay

    # years-since counters evolve mechanically
    if "years_since_hurricane" in out.columns:
        out["years_since_hurricane"] = np.where(out.get("post_maria", 0).astype(float) > 0, out["years_since_hurricane"].astype(float) + 1, out["years_since_hurricane"])
    if "years_since_earthquake" in out.columns:
        out["years_since_earthquake"] = np.where(out.get("post_earthquake_2020", 0).astype(float) > 0, out["years_since_earthquake"].astype(float) + 1, out["years_since_earthquake"])

    return out

def simulate_scenario(latest_rows, scenario_name, scenario, n_sims=N_SIMS, save_detail=SAVE_DETAIL, random_seed=42):
    rng = np.random.default_rng(random_seed)
    base_latest = latest_rows.copy().reset_index(drop=True)

    detail_records = []
    island_records = []
    region_records = []
    muni_records = []

    for sim in range(n_sims):
        sim_df = base_latest.copy()

        for year in FORECAST_YEARS:
            sim_df["year"] = year

            # structural prediction
            X_curr = sim_df[PHASE2_FEATURES].copy()
            structural_pct_3y_avg = phase2_backbone_model.predict(X_curr)
            structural_annual = np.array([predicted_annual_pct_to_decimal(v) for v in structural_pct_3y_avg])

            # explicit event overlay
            annual_overlay = []
            updated_rows = []
            for _, row in sim_df.iterrows():
                row2, adj = apply_scenario_feature_overlay(row, year, scenario)
                annual_overlay.append(adj / 100.0)  # convert pct points to decimal rate
                updated_rows.append(row2)
            sim_df = pd.DataFrame(updated_rows)

            # heteroskedastic random noise calibrated from municipal history and holdout residuals
            noise = np.array([
                rng.normal(
                    0,
                    max(0.001, min(0.03, 0.40 * residual_std / 100.0 + (muni_vol.get(m, global_vol) / 100.0) * 0.50))
                )
                for m in sim_df["municipio"]
            ])

            annual_rate = structural_annual + np.array(annual_overlay) + noise

            prev_pop = sim_df["total_population"].astype(float).values
            next_pop = prev_pop * (1.0 + annual_rate)
            next_pop = np.maximum(next_pop, 0)

            # update lag dynamics and non-lag covariates
            realized_pct = ((next_pop / prev_pop) - 1.0) * 100.0
            sim_df["lag_pop_change_2"] = sim_df["lag_pop_change_1"]
            sim_df["lag_pop_change_1"] = realized_pct
            sim_df["total_population"] = next_pop
            sim_df = evolve_nonlag_covariates(sim_df, realized_pct, year, scenario)

            if save_detail:
                tmp = sim_df[["municipio","region","year","total_population","lag_pop_change_1","lag_pop_change_2","income_growth","establishment_growth"]].copy()
                tmp["scenario"] = scenario_name
                tmp["sim"] = sim
                detail_records.append(tmp)

            island_total = float(sim_df["total_population"].sum())
            island_records.append({"scenario": scenario_name, "sim": sim, "year": year, "population": island_total})

            reg = sim_df.groupby("region", dropna=False)["total_population"].sum().reset_index()
            reg["scenario"] = scenario_name
            reg["sim"] = sim
            reg["year"] = year
            reg = reg.rename(columns={"total_population":"population"})
            region_records.append(reg)

            muni = sim_df[["municipio","region","year","total_population"]].copy()
            muni["scenario"] = scenario_name
            muni["sim"] = sim
            muni = muni.rename(columns={"total_population":"population"})
            muni_records.append(muni)

    detail_df = pd.concat(detail_records, ignore_index=True) if detail_records else pd.DataFrame()
    island_df = pd.DataFrame(island_records)
    region_df = pd.concat(region_records, ignore_index=True)
    muni_df = pd.concat(muni_records, ignore_index=True)
    return detail_df, island_df, region_df, muni_df

def summarize_simulations(df_sim, group_cols):
    q = (
        df_sim.groupby(group_cols + ["year"])["population"]
        .quantile([0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95])
        .unstack()
        .reset_index()
    )
    q.columns = group_cols + ["year", "p05", "p10", "p25", "p50", "p75", "p90", "p95"]
    return q

Residual std from holdout residuals: 1.1857163952576422


In [15]:

# ----------------------------
# Run scenarios
# ----------------------------
scenario_outputs = {}
timing_records = []

import time
for scen_name, scen in scenario_definitions.items():
    t0 = time.time()
    print("Running:", scen_name)
    detail_df, island_df, region_df, muni_df = simulate_scenario(
        latest_rows=latest_rows,
        scenario_name=scen_name,
        scenario=scen,
        n_sims=N_SIMS,
        save_detail=SAVE_DETAIL,
        random_seed=42
    )
    scenario_outputs[scen_name] = {
        "detail": detail_df,
        "island_sim": island_df,
        "region_sim": region_df,
        "muni_sim": muni_df,
        "island_summary": summarize_simulations(island_df, []),
        "region_summary": summarize_simulations(region_df, ["region"]),
        "muni_summary": summarize_simulations(muni_df, ["municipio","region"]),
    }
    elapsed = time.time() - t0
    timing_records.append({"scenario": scen_name, "seconds": elapsed})
    print(f"Done {scen_name} in {elapsed:.2f}s")

timing_df = pd.DataFrame(timing_records)
saved_files.append(save_csv(timing_df, "phase3_v4_scenario_timing.csv"))
timing_df


Running: baseline
Done baseline in 56.41s
Running: hurricane_cat1
Done hurricane_cat1 in 56.41s
Running: hurricane_cat2
Done hurricane_cat2 in 56.52s
Running: hurricane_cat3
Done hurricane_cat3 in 56.43s
Running: hurricane_cat4
Done hurricane_cat4 in 56.49s
Running: hurricane_cat5_maria_like
Done hurricane_cat5_maria_like in 56.30s
Running: earthquake_m6_0
Done earthquake_m6_0 in 56.40s
Running: earthquake_m6_5
Done earthquake_m6_5 in 56.45s
Running: earthquake_m6_8
Done earthquake_m6_8 in 56.33s
Running: earthquake_m7_2
Done earthquake_m7_2 in 58.20s
Running: earthquake_m7_5
Done earthquake_m7_5 in 57.15s
Saved CSV: phase3_outputs_v4/phase3_v4_scenario_timing.csv


,scenario,seconds
0,baseline,56.414098
1,hurricane_cat1,56.409861
2,hurricane_cat2,56.516263
3,hurricane_cat3,56.432042
4,hurricane_cat4,56.485612
5,hurricane_cat5_maria_like,56.299265
6,earthquake_m6_0,56.404244
7,earthquake_m6_5,56.452606
8,earthquake_m6_8,56.327711
9,earthquake_m7_2,58.195900


In [16]:
# ----------------------------
# Maria backtest (2017 event)
# Uses historical rows up to 2017, then simulates a Maria-like path 2018-2021
# ----------------------------
backtest_start_year = 2017
hist_2017 = df[df["year"] == backtest_start_year].copy().sort_values("municipio").reset_index(drop=True)

if hist_2017.empty:
    raise ValueError("Could not build Maria backtest because there are no 2017 rows in the dataset.")

BACKTEST_YEARS = [2018, 2019, 2020, 2021]

def simulate_backtest(latest_rows_2017, scenario, years, n_sims=150, random_seed=42):
    rng = np.random.default_rng(random_seed)
    sim_df0 = latest_rows_2017.copy().reset_index(drop=True)
    island_records = []
    region_records = []

    for sim in range(n_sims):
        sim_df = sim_df0.copy()
        for year in years:
            sim_df["year"] = year
            X_curr = sim_df[PHASE2_FEATURES].copy()
            structural_pct_3y_avg = phase2_backbone_model.predict(X_curr)
            structural_annual = np.array([predicted_annual_pct_to_decimal(v) for v in structural_pct_3y_avg])

            annual_overlay = []
            updated_rows = []
            for _, row in sim_df.iterrows():
                row2, adj = apply_scenario_feature_overlay(row, year, scenario)
                annual_overlay.append(adj / 100.0)
                updated_rows.append(row2)
            sim_df = pd.DataFrame(updated_rows)

            noise = np.array([
                rng.normal(
                    0,
                    max(0.001, min(0.03, 0.40 * residual_std / 100.0 + (muni_vol.get(m, global_vol) / 100.0) * 0.50))
                )
                for m in sim_df["municipio"]
            ])

            annual_rate = structural_annual + np.array(annual_overlay) + noise
            prev_pop = sim_df["total_population"].astype(float).values
            next_pop = np.maximum(prev_pop * (1.0 + annual_rate), 0)
            realized_pct = ((next_pop / prev_pop) - 1.0) * 100.0

            sim_df["lag_pop_change_2"] = sim_df["lag_pop_change_1"]
            sim_df["lag_pop_change_1"] = realized_pct
            sim_df["total_population"] = next_pop
            sim_df = evolve_nonlag_covariates(sim_df, realized_pct, year, scenario)

            island_records.append({
                "sim": sim, "year": year, "population": float(sim_df["total_population"].sum())
            })

            reg = sim_df.groupby("region", dropna=False)["total_population"].sum().reset_index()
            reg["sim"] = sim
            reg["year"] = year
            reg = reg.rename(columns={"total_population":"population"})
            region_records.append(reg)

    island_df = pd.DataFrame(island_records)
    region_df = pd.concat(region_records, ignore_index=True)
    return island_df, region_df

maria_like_scenario_2017 = {"kind":"hurricane","category":5,"event_year":2017}
maria_bt_island_sim, maria_bt_region_sim = simulate_backtest(hist_2017, maria_like_scenario_2017, BACKTEST_YEARS)
maria_bt_island_summary = summarize_simulations(maria_bt_island_sim, [])
maria_bt_region_summary = summarize_simulations(maria_bt_region_sim, ["region"])

# Observed backtest references
obs_island = (
    df[df["year"].between(2010, 2021)]
      .groupby("year", as_index=False)["total_population"].sum()
      .rename(columns={"total_population":"observed_population"})
)
obs_region = (
    df[df["year"].between(2017, 2021)]
      .groupby(["region","year"], as_index=False)["total_population"].sum()
      .rename(columns={"total_population":"observed_population"})
)

maria_bt_island = maria_bt_island_summary.merge(
    obs_island[obs_island["year"].between(2018,2021)], on="year", how="left"
)
maria_bt_region = maria_bt_region_summary.merge(obs_region, on=["region","year"], how="left")

maria_bt_island["error"] = maria_bt_island["p50"] - maria_bt_island["observed_population"]
maria_bt_island["abs_error"] = maria_bt_island["error"].abs()
maria_bt_island["pct_error"] = np.where(
    maria_bt_island["observed_population"].abs() > 0,
    100 * maria_bt_island["error"] / maria_bt_island["observed_population"],
    np.nan
)

maria_backtest_metrics = pd.DataFrame([{
    "backtest_years": f"{BACKTEST_YEARS[0]}-{BACKTEST_YEARS[-1]}",
    "island_mae": float(maria_bt_island["abs_error"].mean()),
    "island_rmse": float(np.sqrt(np.mean(np.square(maria_bt_island["error"])))),
    "island_mape_pct": float(np.nanmean(np.abs(maria_bt_island["pct_error"]))),
    "island_mean_pct_error": float(np.nanmean(maria_bt_island["pct_error"])),
}])

maria_bt_region_eval = maria_bt_region.copy()
maria_bt_region_eval["error"] = maria_bt_region_eval["p50"] - maria_bt_region_eval["observed_population"]
maria_bt_region_eval["abs_error"] = maria_bt_region_eval["error"].abs()
maria_bt_region_eval["pct_error"] = np.where(
    maria_bt_region_eval["observed_population"].abs() > 0,
    100 * maria_bt_region_eval["error"] / maria_bt_region_eval["observed_population"],
    np.nan
)
maria_backtest_region_metrics = (
    maria_bt_region_eval.groupby("region", as_index=False)
    .agg(
        region_mae=("abs_error", "mean"),
        region_rmse=("error", lambda s: float(np.sqrt(np.mean(np.square(s))))),
        region_mape_pct=("pct_error", lambda s: float(np.nanmean(np.abs(s)))),
        region_mean_pct_error=("pct_error", lambda s: float(np.nanmean(s)))
    )
    .sort_values("region_mae", ascending=False)
)

saved_files.append(save_csv(maria_bt_island, "phase3_v4_maria_backtest_island.csv"))
saved_files.append(save_csv(maria_bt_region, "phase3_v4_maria_backtest_region.csv"))
saved_files.append(save_csv(maria_backtest_metrics, "phase3_v4_maria_backtest_metrics.csv"))
saved_files.append(save_csv(maria_backtest_region_metrics, "phase3_v4_maria_backtest_region_metrics.csv"))

maria_backtest_metrics

Saved CSV: phase3_outputs_v4/phase3_v4_maria_backtest_island.csv
Saved CSV: phase3_outputs_v4/phase3_v4_maria_backtest_region.csv
Saved CSV: phase3_outputs_v4/phase3_v4_maria_backtest_metrics.csv
Saved CSV: phase3_outputs_v4/phase3_v4_maria_backtest_region_metrics.csv


,backtest_years,island_mae,island_rmse,island_mape_pct,island_mean_pct_error
0,2018-2021,28913.454153,39518.359719,0.873626,-0.547423


In [17]:

# ----------------------------
# Combine scenario summaries
# ----------------------------
island_summary_all = []
region_summary_all = []
muni_summary_all = []

for scen_name, out in scenario_outputs.items():
    a = out["island_summary"].copy()
    a["scenario"] = scen_name
    island_summary_all.append(a)

    b = out["region_summary"].copy()
    b["scenario"] = scen_name
    region_summary_all.append(b)

    c = out["muni_summary"].copy()
    c["scenario"] = scen_name
    muni_summary_all.append(c)

island_summary_all = pd.concat(island_summary_all, ignore_index=True)
region_summary_all = pd.concat(region_summary_all, ignore_index=True)
muni_summary_all = pd.concat(muni_summary_all, ignore_index=True)

saved_files.append(save_csv(island_summary_all, "phase3_v4_island_scenario_summary.csv"))
saved_files.append(save_csv(region_summary_all, "phase3_v4_region_scenario_summary.csv"))
saved_files.append(save_csv(muni_summary_all, "phase3_v4_municipal_scenario_summary.csv"))

island_summary_all.head()


Saved CSV: phase3_outputs_v4/phase3_v4_island_scenario_summary.csv
Saved CSV: phase3_outputs_v4/phase3_v4_region_scenario_summary.csv
Saved CSV: phase3_outputs_v4/phase3_v4_municipal_scenario_summary.csv


,year,p05,p10,p25,p50,p75,p90,p95,scenario
0,2025,3.177516e+06,3.180590e+06,3.185718e+06,3.189951e+06,3.196387e+06,3.201656e+06,3.203776e+06,baseline
1,2026,3.122489e+06,3.126536e+06,3.133034e+06,3.141569e+06,3.149560e+06,3.156292e+06,3.158790e+06,baseline
2,2027,3.082999e+06,3.085806e+06,3.093836e+06,3.101707e+06,3.109508e+06,3.117845e+06,3.123188e+06,baseline
3,2028,3.039865e+06,3.045038e+06,3.052280e+06,3.061653e+06,3.071019e+06,3.079904e+06,3.085260e+06,baseline
4,2029,2.993528e+06,2.999134e+06,3.008314e+06,3.018629e+06,3.029725e+06,3.040903e+06,3.044885e+06,baseline


In [18]:

# ----------------------------
# Impact summaries relative to baseline
# ----------------------------
baseline_island = island_summary_all[island_summary_all["scenario"] == "baseline"][["year","p50"]].rename(columns={"p50":"baseline_p50"})
baseline_region = region_summary_all[region_summary_all["scenario"] == "baseline"][["region","year","p50"]].rename(columns={"p50":"baseline_p50"})
baseline_muni = muni_summary_all[muni_summary_all["scenario"] == "baseline"][["municipio","region","year","p50"]].rename(columns={"p50":"baseline_p50"})

island_impact = island_summary_all.merge(baseline_island, on="year", how="left")
island_impact["abs_impact_vs_baseline"] = island_impact["p50"] - island_impact["baseline_p50"]
island_impact["pct_impact_vs_baseline"] = 100 * island_impact["abs_impact_vs_baseline"] / island_impact["baseline_p50"]

region_impact = region_summary_all.merge(baseline_region, on=["region","year"], how="left")
region_impact["abs_impact_vs_baseline"] = region_impact["p50"] - region_impact["baseline_p50"]
region_impact["pct_impact_vs_baseline"] = 100 * region_impact["abs_impact_vs_baseline"] / region_impact["baseline_p50"]

muni_impact = muni_summary_all.merge(baseline_muni, on=["municipio","region","year"], how="left")
muni_impact["abs_impact_vs_baseline"] = muni_impact["p50"] - muni_impact["baseline_p50"]
muni_impact["pct_impact_vs_baseline"] = 100 * muni_impact["abs_impact_vs_baseline"] / muni_impact["baseline_p50"]

saved_files.append(save_csv(island_impact, "phase3_v4_island_impact_summary.csv"))
saved_files.append(save_csv(region_impact, "phase3_v4_region_impact_summary.csv"))

region_2030 = region_impact[region_impact["year"] == 2030].copy()
muni_2030 = muni_impact[muni_impact["year"] == 2030].copy()

saved_files.append(save_csv(region_2030, "phase3_v4_region_2030_comparison.csv"))
saved_files.append(save_csv(muni_2030, "phase3_v4_municipal_2030_comparison.csv"))

island_impact.head()


Saved CSV: phase3_outputs_v4/phase3_v4_island_impact_summary.csv
Saved CSV: phase3_outputs_v4/phase3_v4_region_impact_summary.csv
Saved CSV: phase3_outputs_v4/phase3_v4_region_2030_comparison.csv
Saved CSV: phase3_outputs_v4/phase3_v4_municipal_2030_comparison.csv


,year,p05,p10,p25,p50,p75,p90,p95,scenario,baseline_p50,abs_impact_vs_baseline,pct_impact_vs_baseline
0,2025,3.177516e+06,3.180590e+06,3.185718e+06,3.189951e+06,3.196387e+06,3.201656e+06,3.203776e+06,baseline,3.189951e+06,0.0,0.0
1,2026,3.122489e+06,3.126536e+06,3.133034e+06,3.141569e+06,3.149560e+06,3.156292e+06,3.158790e+06,baseline,3.141569e+06,0.0,0.0
2,2027,3.082999e+06,3.085806e+06,3.093836e+06,3.101707e+06,3.109508e+06,3.117845e+06,3.123188e+06,baseline,3.101707e+06,0.0,0.0
3,2028,3.039865e+06,3.045038e+06,3.052280e+06,3.061653e+06,3.071019e+06,3.079904e+06,3.085260e+06,baseline,3.061653e+06,0.0,0.0
4,2029,2.993528e+06,2.999134e+06,3.008314e+06,3.018629e+06,3.029725e+06,3.040903e+06,3.044885e+06,baseline,3.018629e+06,0.0,0.0


In [19]:
# ----------------------------
# Scenario severity ranking tables (2030 impact vs baseline)
# ----------------------------
island_2030_rank = (
    island_impact[island_impact["year"] == 2030]
    .loc[lambda d: d["scenario"] != "baseline", ["scenario", "p50", "baseline_p50", "abs_impact_vs_baseline", "pct_impact_vs_baseline"]]
    .sort_values("abs_impact_vs_baseline")
    .reset_index(drop=True)
)

region_2030_rank = (
    region_impact[region_impact["year"] == 2030]
    .loc[lambda d: d["scenario"] != "baseline", ["scenario", "region", "p50", "baseline_p50", "abs_impact_vs_baseline", "pct_impact_vs_baseline"]]
    .sort_values(["scenario", "abs_impact_vs_baseline"])
    .reset_index(drop=True)
)

muni_2030_rank = (
    muni_impact[muni_impact["year"] == 2030]
    .loc[lambda d: d["scenario"] != "baseline", ["scenario", "municipio", "region", "p50", "baseline_p50", "abs_impact_vs_baseline", "pct_impact_vs_baseline"]]
    .sort_values(["scenario", "abs_impact_vs_baseline"])
    .reset_index(drop=True)
)

saved_files.append(save_csv(island_2030_rank, "phase3_v4_island_2030_severity_rank.csv"))
saved_files.append(save_csv(region_2030_rank, "phase3_v4_region_2030_severity_rank.csv"))
saved_files.append(save_csv(muni_2030_rank, "phase3_v4_municipio_2030_severity_rank.csv"))

print("Island 2030 severity ranking:")
display(island_2030_rank.head(10))

Saved CSV: phase3_outputs_v4/phase3_v4_island_2030_severity_rank.csv
Saved CSV: phase3_outputs_v4/phase3_v4_region_2030_severity_rank.csv
Saved CSV: phase3_outputs_v4/phase3_v4_municipio_2030_severity_rank.csv
Island 2030 severity ranking:


,scenario,p50,baseline_p50,abs_impact_vs_baseline,pct_impact_vs_baseline
0,hurricane_cat5_maria_like,2.856362e+06,2.976181e+06,-119819.922956,-4.025962
1,hurricane_cat4,2.898856e+06,2.976181e+06,-77325.623853,-2.598149
2,earthquake_m7_2,2.921058e+06,2.976181e+06,-55123.589253,-1.852158
3,earthquake_m7_5,2.921058e+06,2.976181e+06,-55123.589253,-1.852158
4,hurricane_cat3,2.929523e+06,2.976181e+06,-46658.610906,-1.567734
5,earthquake_m6_8,2.933791e+06,2.976181e+06,-42390.640995,-1.424330
6,earthquake_m6_5,2.946532e+06,2.976181e+06,-29649.188080,-0.996216
7,hurricane_cat2,2.950575e+06,2.976181e+06,-25606.668969,-0.860387
8,earthquake_m6_0,2.958471e+06,2.976181e+06,-17710.824315,-0.595085
9,hurricane_cat1,2.962357e+06,2.976181e+06,-13824.144576,-0.464493


## Scenario-summary note

The impact tables below rank scenarios by **2030 median population effect relative to baseline** at the island, region, and municipality levels.  
These tables are useful for slides because they summarize the scenario engine in one place and make it easier to compare the severity ordering across hazards.

In [20]:

# ----------------------------
# Plot helpers
# ----------------------------
observed_island = (
    df.groupby("year", as_index=False)["total_population"].sum()
      .rename(columns={"total_population":"population"})
)

def add_chart_record(filename, title, description):
    chart_index_records.append({
        "filename": filename,
        "title": title,
        "description": description
    })

def plot_fan(ax, summary_df, label_prefix="", median_color=None, fill_alpha=0.15, add_label=True):
    ax.fill_between(summary_df["year"], summary_df["p05"], summary_df["p95"], alpha=fill_alpha)
    ax.fill_between(summary_df["year"], summary_df["p10"], summary_df["p90"], alpha=fill_alpha + 0.05)
    ax.fill_between(summary_df["year"], summary_df["p25"], summary_df["p75"], alpha=fill_alpha + 0.08)
    ax.plot(summary_df["year"], summary_df["p50"], linewidth=2)

def style_observed(ax):
    ax.plot(observed_island["year"], observed_island["population"], marker="o", linewidth=2, label="Observed")
    ax.axvline(latest_year, linestyle="--", linewidth=1.5)
    ax.set_xlabel("Year")
    ax.set_ylabel("Population")
    ax.grid(True, alpha=0.3)


In [21]:

# ----------------------------
# Chart 1: all scenario island fan charts
# ----------------------------
scens = ["baseline","hurricane_cat3","hurricane_cat5_maria_like","earthquake_m6_5","earthquake_m6_8","earthquake_m7_5"]
scens = [s for s in scens if s in island_summary_all["scenario"].unique()]

n = len(scens)
fig, axes = plt.subplots(2, math.ceil(n/2), figsize=(18, 8), sharey=True)
axes = np.array(axes).reshape(-1)

for ax, scen in zip(axes, scens):
    style_observed(ax)
    s = island_summary_all[island_summary_all["scenario"] == scen].sort_values("year")
    plot_fan(ax, s)
    ax.set_title(scen.replace("_"," ").title())

for ax in axes[n:]:
    ax.axis("off")

fig.suptitle("Island-level fan charts by scenario", fontsize=16)
fname = "phase3_v4_island_fan_charts_all_scenarios.png"
saved_files.append(save_fig(fig, fname))
add_chart_record(fname, "Island fan charts by scenario", "Observed island history with forecast fan charts for baseline and selected hazard scenarios.")


Saved FIG: phase3_outputs_v4/phase3_v4_island_fan_charts_all_scenarios.png


In [22]:

# ----------------------------
# Chart 2: baseline vs hurricane ladder
# ----------------------------
h_ladder = [s for s in ["baseline","hurricane_cat1","hurricane_cat2","hurricane_cat3","hurricane_cat4","hurricane_cat5_maria_like"] if s in island_summary_all["scenario"].unique()]
fig, ax = plt.subplots(figsize=(12, 6))
style_observed(ax)
for scen in h_ladder:
    s = island_summary_all[island_summary_all["scenario"] == scen].sort_values("year")
    ax.plot(s["year"], s["p50"], linewidth=2, label=scen.replace("_"," ").title())
ax.legend()
ax.set_title("Island median forecast paths — hurricane magnitude ladder")
fname = "phase3_v4_island_hurricane_ladder.png"
saved_files.append(save_fig(fig, fname))
add_chart_record(fname, "Island hurricane ladder", "Median island forecast paths comparing baseline against Cat 1 through Cat 5 Maria-like hurricane scenarios.")


Saved FIG: phase3_outputs_v4/phase3_v4_island_hurricane_ladder.png


In [23]:

# ----------------------------
# Chart 3: baseline vs earthquake ladder
# ----------------------------
e_ladder = [s for s in ["baseline","earthquake_m6_0","earthquake_m6_5","earthquake_m6_8","earthquake_m7_2","earthquake_m7_5"] if s in island_summary_all["scenario"].unique()]
fig, ax = plt.subplots(figsize=(12, 6))
style_observed(ax)
for scen in e_ladder:
    s = island_summary_all[island_summary_all["scenario"] == scen].sort_values("year")
    ax.plot(s["year"], s["p50"], linewidth=2, label=scen.replace("_"," ").title())
ax.legend()
ax.set_title("Island median forecast paths — earthquake magnitude ladder")
fname = "phase3_v4_island_earthquake_ladder.png"
saved_files.append(save_fig(fig, fname))
add_chart_record(fname, "Island earthquake ladder", "Median island forecast paths comparing baseline against multiple earthquake magnitude scenarios.")


Saved FIG: phase3_outputs_v4/phase3_v4_island_earthquake_ladder.png


In [24]:

# ----------------------------
# Chart 4: side-by-side baseline vs selected hazards
# ----------------------------
pairs = [("hurricane_cat3", "No disaster vs Hurricane Cat 3"),
         ("hurricane_cat5_maria_like", "No disaster vs Cat 5 Maria-like"),
         ("earthquake_m6_8", "No disaster vs Earthquake M6.8"),
         ("earthquake_m7_5", "No disaster vs Earthquake M7.5")]

fig, axes = plt.subplots(2, 2, figsize=(16, 10), sharey=True)
axes = axes.flatten()
base = island_summary_all[island_summary_all["scenario"] == "baseline"].sort_values("year")

for ax, (scen, title) in zip(axes, pairs):
    style_observed(ax)
    plot_fan(ax, base)
    if scen in island_summary_all["scenario"].unique():
        s = island_summary_all[island_summary_all["scenario"] == scen].sort_values("year")
        ax.plot(base["year"], base["p50"], linestyle="--", linewidth=2, label="Baseline median")
        ax.plot(s["year"], s["p50"], linewidth=2.5, label="Scenario median")
    ax.set_title(title)
    ax.legend()

fig.suptitle("Baseline vs selected hazard scenarios", fontsize=16)
fname = "phase3_v4_side_by_side_baseline_vs_hazards.png"
saved_files.append(save_fig(fig, fname))
add_chart_record(fname, "Baseline vs selected hazards", "Side-by-side island-level comparison panels for baseline against selected hurricane and earthquake scenarios.")


Saved FIG: phase3_outputs_v4/phase3_v4_side_by_side_baseline_vs_hazards.png


In [25]:

# ----------------------------
# Chart 5: island impact over time relative to baseline
# ----------------------------
fig, ax = plt.subplots(figsize=(12, 6))
for scen in [s for s in island_impact["scenario"].unique() if s != "baseline"]:
    s = island_impact[island_impact["scenario"] == scen].sort_values("year")
    ax.plot(s["year"], s["abs_impact_vs_baseline"], linewidth=2, label=scen.replace("_"," ").title())
ax.axhline(0, color="black", linewidth=1)
ax.set_title("Island median impact vs baseline over time")
ax.set_xlabel("Year")
ax.set_ylabel("Population difference vs baseline")
ax.grid(True, alpha=0.3)
ax.legend(ncol=2)
fname = "phase3_v4_island_impact_over_time.png"
saved_files.append(save_fig(fig, fname))
add_chart_record(fname, "Island impact over time", "Difference between scenario median and baseline median island population by year.")


Saved FIG: phase3_outputs_v4/phase3_v4_island_impact_over_time.png


In [26]:

# ----------------------------
# Chart 6: Maria backtest island
# ----------------------------
fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(obs_island["year"], obs_island["observed_population"], marker="o", linewidth=2, label="Observed island population")
ax.plot(maria_bt_island["year"], maria_bt_island["p50"], marker="o", linewidth=2, label="Maria-like model path")
ax.axvline(2017, color="red", linestyle="--", linewidth=1.5, label="Maria year")
ax.set_title("Maria backtest: observed vs Maria-like modeled island path")
ax.set_xlabel("Year")
ax.set_ylabel("Population")
ax.grid(True, alpha=0.3)
ax.legend()
fname = "phase3_v4_maria_backtest_island.png"
saved_files.append(save_fig(fig, fname))
add_chart_record(fname, "Maria backtest island", "Observed island population compared with Maria-like modeled backtest path from 2018-2021.")


Saved FIG: phase3_outputs_v4/phase3_v4_maria_backtest_island.png


In [27]:

# ----------------------------
# Chart 7: region 2030 impact bar charts for selected scenarios
# ----------------------------
selected_scenarios = [s for s in ["hurricane_cat3","hurricane_cat5_maria_like","earthquake_m6_8","earthquake_m7_5"] if s in region_2030["scenario"].unique()]
fig, axes = plt.subplots(2, 2, figsize=(16, 10), sharey=True)
axes = axes.flatten()

for ax, scen in zip(axes, selected_scenarios):
    s = region_2030[region_2030["scenario"] == scen].sort_values("abs_impact_vs_baseline")
    ax.barh(s["region"], s["abs_impact_vs_baseline"])
    ax.set_title(f"2030 region impact vs baseline — {scen.replace('_',' ').title()}")
    ax.set_xlabel("Population difference vs baseline")

for ax in axes[len(selected_scenarios):]:
    ax.axis("off")

fname = "phase3_v4_region_2030_impact_bars.png"
saved_files.append(save_fig(fig, fname))
add_chart_record(fname, "Region 2030 impact bars", "Horizontal bar charts showing region-level 2030 population impact versus baseline for selected scenarios.")


Saved FIG: phase3_outputs_v4/phase3_v4_region_2030_impact_bars.png


In [28]:

# ----------------------------
# Chart 8: regional faceted median paths for Maria-like and Cat 3
# ----------------------------
faceted_scenarios = [s for s in ["baseline","hurricane_cat3","hurricane_cat5_maria_like"] if s in region_summary_all["scenario"].unique()]
regions = [r for r in sorted(region_summary_all["region"].dropna().unique()) if r != "Unknown"]
n = len(regions)
fig, axes = plt.subplots(math.ceil(n/3), 3, figsize=(18, 4*math.ceil(n/3)), sharex=True, sharey=False)
axes = np.array(axes).reshape(-1)

for ax, region in zip(axes, regions):
    obs_r = df[df["region"] == region].groupby("year", as_index=False)["total_population"].sum()
    ax.plot(obs_r["year"], obs_r["total_population"], marker="o", linewidth=2, label="Observed")
    ax.axvline(latest_year, linestyle="--", linewidth=1.2)
    for scen in faceted_scenarios:
        s = region_summary_all[(region_summary_all["region"] == region) & (region_summary_all["scenario"] == scen)].sort_values("year")
        ax.plot(s["year"], s["p50"], linewidth=2, label=scen.replace("_"," ").title())
    ax.set_title(region)
    ax.grid(True, alpha=0.3)

for ax in axes[n:]:
    ax.axis("off")

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper center", ncol=4)
fig.suptitle("Regional median paths — baseline, Cat 3, and Cat 5 Maria-like", fontsize=16, y=0.98)
fname = "phase3_v4_region_faceted_paths.png"
saved_files.append(save_fig(fig, fname))
add_chart_record(fname, "Regional faceted paths", "Faceted regional median path charts comparing baseline, hurricane Cat 3, and Cat 5 Maria-like scenarios.")


Saved FIG: phase3_outputs_v4/phase3_v4_region_faceted_paths.png


In [29]:

# ----------------------------
# Chart 9: municipal 2030 top losses under Maria-like
# ----------------------------
if "hurricane_cat5_maria_like" in muni_2030["scenario"].unique():
    s = muni_2030[muni_2030["scenario"] == "hurricane_cat5_maria_like"].sort_values("abs_impact_vs_baseline").head(15)
    fig, ax = plt.subplots(figsize=(12, 7))
    ax.barh(s["municipio"], s["abs_impact_vs_baseline"])
    ax.set_title("Top municipal 2030 losses vs baseline — Cat 5 Maria-like")
    ax.set_xlabel("Population difference vs baseline")
    fname = "phase3_v4_top_municipal_losses_maria_like.png"
    saved_files.append(save_fig(fig, fname))
    add_chart_record(fname, "Top municipal losses Maria-like", "Municipalities with the largest modeled 2030 population losses relative to baseline under a Cat 5 Maria-like hurricane.")


Saved FIG: phase3_outputs_v4/phase3_v4_top_municipal_losses_maria_like.png


In [30]:

# ----------------------------
# Chart 10: region heatmap for cumulative impact (2026-2030)
# ----------------------------
heat = (
    region_impact[region_impact["scenario"].isin([s for s in selected_scenarios])]
    .groupby(["scenario","region","year"], as_index=False)["abs_impact_vs_baseline"].mean()
)
for scen in heat["scenario"].unique():
    pivot = heat[heat["scenario"] == scen].pivot(index="region", columns="year", values="abs_impact_vs_baseline")
    fig, ax = plt.subplots(figsize=(10, 5))
    im = ax.imshow(pivot.values, aspect="auto")
    ax.set_xticks(range(len(pivot.columns)))
    ax.set_xticklabels(pivot.columns)
    ax.set_yticks(range(len(pivot.index)))
    ax.set_yticklabels(pivot.index)
    ax.set_title(f"Region impact heatmap — {scen.replace('_',' ').title()}")
    ax.set_xlabel("Year")
    ax.set_ylabel("Region")
    fig.colorbar(im, ax=ax, label="Population difference vs baseline")
    fname = f"phase3_v4_heatmap_{scen}.png"
    saved_files.append(save_fig(fig, fname))
    add_chart_record(fname, f"Heatmap {scen}", f"Regional impact heatmap over forecast years for {scen.replace('_',' ')}.")


Saved FIG: phase3_outputs_v4/phase3_v4_heatmap_earthquake_m6_8.png
Saved FIG: phase3_outputs_v4/phase3_v4_heatmap_earthquake_m7_5.png
Saved FIG: phase3_outputs_v4/phase3_v4_heatmap_hurricane_cat3.png
Saved FIG: phase3_outputs_v4/phase3_v4_heatmap_hurricane_cat5_maria_like.png


In [31]:

# ----------------------------
# Chart 11: uncertainty width comparison
# ----------------------------
unc = island_summary_all.copy()
unc["width_90"] = unc["p95"] - unc["p05"]
fig, ax = plt.subplots(figsize=(12, 6))
for scen in [s for s in island_summary_all["scenario"].unique()]:
    s = unc[unc["scenario"] == scen].sort_values("year")
    ax.plot(s["year"], s["width_90"], linewidth=2, label=scen.replace("_"," ").title())
ax.set_title("Island uncertainty width over time (95%-5%)")
ax.set_xlabel("Year")
ax.set_ylabel("Width of uncertainty band")
ax.grid(True, alpha=0.3)
ax.legend(ncol=2)
fname = "phase3_v4_uncertainty_width_comparison.png"
saved_files.append(save_fig(fig, fname))
add_chart_record(fname, "Uncertainty width comparison", "Comparison of island forecast uncertainty band width over time across scenarios.")


Saved FIG: phase3_outputs_v4/phase3_v4_uncertainty_width_comparison.png


## Additional presentation-ready charts and diagnostics

These charts improve the notebook for slide generation by showing:
- modeled vs observed Maria backtest error,
- percent impact relative to baseline,
- a region-by-scenario 2030 heatmap,
- vulnerability vs modeled impact at the municipal level,
- and direct comparison of top municipal losses across major hazards.

These are meant to complement the existing island, region, and municipal scenario figures.


In [32]:

# ----------------------------
# Chart 12: Maria backtest error / deviation from observed
# ----------------------------
# maria_bt_island already contains observed_population from the earlier backtest merge.
# Re-merge only if that column is not present. If a re-merge creates suffixes, normalize them.
bt_compare = maria_bt_island.copy()

if "observed_population" not in bt_compare.columns:
    bt_compare = bt_compare.merge(
        obs_island[["year", "observed_population"]],
        on="year",
        how="left",
        suffixes=("", "_obs")
    ).copy()

# Normalize any suffixed observed-population columns that may have appeared after a merge.
if "observed_population" not in bt_compare.columns:
    for cand in ["observed_population_obs", "observed_population_x", "observed_population_y"]:
        if cand in bt_compare.columns:
            bt_compare["observed_population"] = bt_compare[cand]
            break

if "observed_population" not in bt_compare.columns:
    raise KeyError(
        "Could not find observed_population in Maria backtest data. "
        f"Available columns: {list(bt_compare.columns)}"
    )

bt_compare = bt_compare[bt_compare["year"].between(2018, 2021)].copy()
bt_compare["model_minus_observed"] = bt_compare["p50"] - bt_compare["observed_population"]
bt_compare["pct_diff_vs_observed"] = np.where(
    bt_compare["observed_population"].abs() > 0,
    100 * bt_compare["model_minus_observed"] / bt_compare["observed_population"],
    np.nan
)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
axes[0].bar(bt_compare["year"].astype(str), bt_compare["model_minus_observed"])
axes[0].axhline(0, color="black", linewidth=1)
axes[0].set_title("Maria backtest error by year")
axes[0].set_ylabel("Modeled - observed population")

axes[1].bar(bt_compare["year"].astype(str), bt_compare["pct_diff_vs_observed"])
axes[1].axhline(0, color="black", linewidth=1)
axes[1].set_title("Maria backtest percent error by year")
axes[1].set_ylabel("% difference vs observed")

fname = "phase3_v4_maria_backtest_error.png"
saved_files.append(save_fig(fig, fname))
add_chart_record(
    fname,
    "Maria backtest error",
    "Two-panel chart showing the Maria-like island backtest deviation from observed population in levels and percent terms."
)


Saved FIG: phase3_outputs_v4/phase3_v4_maria_backtest_error.png


In [33]:

# ----------------------------
# Chart 13: island percent impact relative to baseline
# ----------------------------
impact_pct = island_impact.copy()
impact_pct["pct_impact_vs_baseline"] = np.where(
    impact_pct["baseline_p50"].abs() > 0,
    100 * impact_pct["abs_impact_vs_baseline"] / impact_pct["baseline_p50"],
    np.nan
)

fig, ax = plt.subplots(figsize=(12, 6))
focus_scenarios = [s for s in ["hurricane_cat3", "hurricane_cat5_maria_like", "earthquake_m6_8", "earthquake_m7_5"] if s in impact_pct["scenario"].unique()]
for scen in focus_scenarios:
    s = impact_pct[impact_pct["scenario"] == scen].sort_values("year")
    ax.plot(s["year"], s["pct_impact_vs_baseline"], linewidth=2, label=scen.replace("_", " ").title())
ax.axhline(0, color="black", linewidth=1)
ax.set_title("Island impact relative to baseline over time")
ax.set_xlabel("Year")
ax.set_ylabel("% difference vs baseline population")
ax.grid(True, alpha=0.3)
ax.legend()

fname = "phase3_v4_island_pct_impact_over_time.png"
saved_files.append(save_fig(fig, fname))
add_chart_record(fname, "Island percent impact over time", "Percent difference between scenario median and baseline median island population over forecast years.")


Saved FIG: phase3_outputs_v4/phase3_v4_island_pct_impact_over_time.png


In [34]:

# ----------------------------
# Chart 14: region x scenario heatmap for 2030 impact
# ----------------------------
heat2030 = region_2030.copy()
if not heat2030.empty:
    scen_order = [s for s in ["hurricane_cat1","hurricane_cat2","hurricane_cat3","hurricane_cat4","hurricane_cat5_maria_like","earthquake_m6_0","earthquake_m6_5","earthquake_m6_8","earthquake_m7_2","earthquake_m7_5"] if s in heat2030["scenario"].unique()]
    pivot = heat2030.pivot(index="region", columns="scenario", values="abs_impact_vs_baseline")
    pivot = pivot[[c for c in scen_order if c in pivot.columns]] if scen_order else pivot
    fig, ax = plt.subplots(figsize=(14, 6))
    im = ax.imshow(pivot.values, aspect="auto")
    ax.set_xticks(range(len(pivot.columns)))
    ax.set_xticklabels([c.replace("_", " ").title() for c in pivot.columns], rotation=45, ha="right")
    ax.set_yticks(range(len(pivot.index)))
    ax.set_yticklabels(pivot.index)
    ax.set_title("Region-by-scenario 2030 impact heatmap")
    ax.set_xlabel("Scenario")
    ax.set_ylabel("Region")
    cbar = fig.colorbar(im, ax=ax)
    cbar.set_label("Population difference vs baseline")
    fname = "phase3_v4_region_scenario_heatmap_2030.png"
    saved_files.append(save_fig(fig, fname))
    add_chart_record(fname, "Region-scenario heatmap 2030", "Heatmap of region-level 2030 population impact versus baseline across hurricane and earthquake magnitude scenarios.")


Saved FIG: phase3_outputs_v4/phase3_v4_region_scenario_heatmap_2030.png


In [35]:

# ----------------------------
# Chart 15: municipal vulnerability vs impact scatter (Maria-like 2030)
# ----------------------------
if "hurricane_cat5_maria_like" in muni_2030["scenario"].unique():
    latest_baseline = (
        df.sort_values(["municipio", "year"])
          .groupby("municipio", as_index=False)
          .tail(1)[["municipio", "region", "svi_pca_1", "baseline_vulnerability_index", "total_population"]]
          .drop_duplicates("municipio")
    )
    scat = muni_2030[muni_2030["scenario"] == "hurricane_cat5_maria_like"].merge(
        latest_baseline, on="municipio", how="left", suffixes=("", "_base")
    ).copy()

    # Normalize possible region column naming after merge
    if "region" not in scat.columns:
        for cand in ["region_base", "region_x", "region_y"]:
            if cand in scat.columns:
                scat["region"] = scat[cand]
                break

    # Last-resort region mapping from the master dataframe
    if "region" not in scat.columns:
        region_lookup = (
            df[["municipio", "region"]]
            .dropna()
            .drop_duplicates("municipio")
        )
        scat = scat.merge(region_lookup, on="municipio", how="left", suffixes=("", "_lookup"))
        if "region" not in scat.columns and "region_lookup" in scat.columns:
            scat["region"] = scat["region_lookup"]

    # Pick vulnerability column robustly
    if "svi_pca_1" in scat.columns:
        xcol = "svi_pca_1"
    elif "baseline_vulnerability_index" in scat.columns:
        xcol = "baseline_vulnerability_index"
    else:
        raise KeyError("Could not find a vulnerability column for the municipal scatter plot.")

    fig, ax = plt.subplots(figsize=(11, 7))
    group_col = "region" if "region" in scat.columns else None

    if group_col is not None:
        for region_name, grp in scat.groupby(group_col, dropna=False):
            label = region_name if pd.notna(region_name) else "Unknown region"
            ax.scatter(grp[xcol], grp["abs_impact_vs_baseline"], label=label, alpha=0.75)
        ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
    else:
        ax.scatter(scat[xcol], scat["abs_impact_vs_baseline"], alpha=0.75)

    ax.axhline(0, color="black", linewidth=1)
    ax.set_title("Municipal vulnerability vs modeled 2030 impact — Cat 5 Maria-like")
    ax.set_xlabel("Vulnerability metric")
    ax.set_ylabel("Population difference vs baseline")
    ax.grid(True, alpha=0.3)
    fname = "phase3_v4_municipal_vulnerability_vs_impact_scatter.png"
    saved_files.append(save_fig(fig, fname))
    add_chart_record(fname, "Municipal vulnerability vs impact", "Scatter plot showing whether more vulnerable municipalities experience larger modeled 2030 losses under the Cat 5 Maria-like scenario.")


Saved FIG: phase3_outputs_v4/phase3_v4_municipal_vulnerability_vs_impact_scatter.png


In [36]:

# ----------------------------
# Chart 16: top municipal losses comparison (Maria-like vs M7.5)
# ----------------------------
compare_scenarios = [s for s in ["hurricane_cat5_maria_like", "earthquake_m7_5"] if s in muni_2030["scenario"].unique()]
if compare_scenarios:
    top_rows = []
    for scen in compare_scenarios:
        tmp = (
            muni_2030[muni_2030["scenario"] == scen]
            .sort_values("abs_impact_vs_baseline")
            .head(10)
            .copy()
        )
        tmp["scenario_label"] = scen.replace("_", " ").title()
        top_rows.append(tmp)
    top_compare = pd.concat(top_rows, ignore_index=True)
    fig, axes = plt.subplots(1, len(compare_scenarios), figsize=(7*len(compare_scenarios), 7), sharex=False)
    if len(compare_scenarios) == 1:
        axes = [axes]
    for ax, scen in zip(axes, compare_scenarios):
        s = top_compare[top_compare["scenario_label"] == scen.replace("_", " ").title()].sort_values("abs_impact_vs_baseline")
        ax.barh(s["municipio"], s["abs_impact_vs_baseline"])
        ax.set_title(f"Top municipal losses — {scen.replace('_', ' ').title()}")
        ax.set_xlabel("Population difference vs baseline")
    fname = "phase3_v4_top_municipal_losses_compare.png"
    saved_files.append(save_fig(fig, fname))
    add_chart_record(fname, "Top municipal losses comparison", "Side-by-side comparison of the municipalities with the largest 2030 losses under the Maria-like hurricane and M7.5 earthquake scenarios.")


Saved FIG: phase3_outputs_v4/phase3_v4_top_municipal_losses_compare.png


## Conclusion

This Phase 3 notebook extends the Phase 2 modeling workflow into a multiscale scenario-forecasting framework for Puerto Rico. The forecasting engine combines an Extra Trees backbone derived from Phase 2 modeling with stylized hazard overlays for hurricanes and earthquakes, then simulates municipal, regional, and island paths through 2030.

### What was improved in this version

- Uncertainty bands now use **holdout residuals (validation + test)** rather than training residuals, making intervals more conservative.
- Non-lag drivers such as **income growth** and **establishment growth** now evolve through simple persistence and mean-reversion rules rather than remaining fully static.
- The notebook now includes **numeric Maria backtest metrics** at both island and region levels.
- New severity tables rank **2030 scenario impacts** at island, region, and municipality scales.
- The notebook explicitly documents that the scenario shocks are **stylized overlays**, not causal estimates.
- The forecasting backbone now loads the **exact fitted Phase 2 pipeline**, ensuring consistency and eliminating reconstruction drift across phases.

### Model Performance and Validation

The Phase 3 backbone demonstrates **improved and stable generalization performance** relative to Phase 2:

- Validation R² increased substantially
- Test R² improved from approximately **0.18 (Phase 2)** to **0.23 (Phase 3)**
- The reproduction gap between Phase 2 and Phase 3 results is small, indicating strong consistency across phases

These results confirm that the forecasting backbone is **robust and reliable**, providing a solid foundation for scenario analysis.

### How to interpret the results

- The **baseline path** should be read as a conditional forecast based on the model backbone and current structural patterns in the data.
- The **hurricane and earthquake scenarios** are best interpreted as stress tests showing how population paths may diverge under different hazard intensities.
- Municipal-level outputs are useful for identifying relative exposure and vulnerability, but they are more uncertain than region- or island-level summaries.
- Because the model demonstrates stable performance, the forecasts can be interpreted as **credible projections of directional trends**, while still acknowledging uncertainty at finer geographic scales.

### Key Insight

The forecasting results highlight that population dynamics in Puerto Rico are strongly driven by structural persistence, while remaining highly sensitive to external shocks. Scenario comparisons show that hurricane and earthquake events can produce meaningful deviations from baseline population trajectories over time, particularly in more vulnerable regions.

Importantly, the improved model performance in Phase 3 indicates that these projections are grounded in a **validated and consistent predictive framework**, strengthening confidence in the scenario results. While uncertainty remains—especially at the municipal level—the model provides a reliable basis for understanding how structural trends and external shocks interact to shape future population dynamics.

### Future Refinement

For further refinement, future work could incorporate additional drivers such as migration flows, policy interventions, and recovery dynamics, as well as explore alternative modeling approaches (e.g., spatial or time-series models). These enhancements could improve predictive precision and deepen understanding of localized population changes.

In [37]:

# ----------------------------
# Chart index / manifest
# ----------------------------
chart_index_df = pd.DataFrame(chart_index_records)
manifest_df = pd.DataFrame({"filepath": saved_files})

saved_files.append(save_csv(chart_index_df, "phase3_v4_chart_index.csv"))
saved_files.append(save_csv(manifest_df, "phase3_v4_saved_files_manifest.csv"))

chart_index_df.head()


Saved CSV: phase3_outputs_v4/phase3_v4_chart_index.csv
Saved CSV: phase3_outputs_v4/phase3_v4_saved_files_manifest.csv


,filename,title,description
0,phase3_v4_island_fan_charts_all_scenarios.png,Island fan charts by scenario,Observed island history with forecast fan char...
1,phase3_v4_island_hurricane_ladder.png,Island hurricane ladder,Median island forecast paths comparing baselin...
2,phase3_v4_island_earthquake_ladder.png,Island earthquake ladder,Median island forecast paths comparing baselin...
3,phase3_v4_side_by_side_baseline_vs_hazards.png,Baseline vs selected hazards,Side-by-side island-level comparison panels fo...
4,phase3_v4_island_impact_over_time.png,Island impact over time,Difference between scenario median and baselin...
